# Top 20% decline prediction: feature engineering

**Population**: households that were in the top 20% on the early window (weeks 17-26) (about 499)
**Label**: did the household stay in the top 20% on the late window (weeks 90-99) too (0 = stayed) vs did not (1 = declined/dropped out); definition (b)

**Features used (only those settled by the EDA)**
- Derived from transaction_data: early-window spending, visit frequency, category diversity, mean items per basket
- causal_data: share of early-window purchases that were of display/mailer-exposed products
- hh_demographic: original demographic values (missing values included as a "No record" category)

**Excluded (with the reason found in the EDA)**
- coupon_redempt, campaign_table: there is no activity at all in the early window (DAY 111-180), so they are all constant (0) and carry no information

**All features use only the early window (weeks 17-26), to prevent data leakage**

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)

from config import DATA_DIR, OUT_DIR, INTERMEDIATE_DIR, FIG_DIR, SEED

STABLE_MIN_WEEK = 17
STABLE_MAX_WEEK = 99
EARLY_N = 10
TOP_X_PCT = 20

tx = pd.read_csv(DATA_DIR / "transaction_data.csv")
hh_demo = pd.read_csv(DATA_DIR / "hh_demographic.csv")
causal = pd.read_csv(DATA_DIR / "causal_data.csv")
product = pd.read_csv(DATA_DIR / "product.csv")

# Unify the column-name case (avoids the issue where causal_data was lower case)
causal.columns = [c.upper() for c in causal.columns]
tx.columns = [c.upper() for c in tx.columns]
hh_demo.columns = [c.upper() for c in hh_demo.columns]
product.columns = [c.upper() for c in product.columns]

tx_stable = tx[(tx["WEEK_NO"] >= STABLE_MIN_WEEK) & (tx["WEEK_NO"] <= STABLE_MAX_WEEK)].copy()
stable_households = np.sort(tx_stable["HOUSEHOLD_KEY"].unique())
print(f"Households in the stable window: {len(stable_households):,}")


Households in the stable window: 2,492


## 1. Re-confirm the early top 20% households (the labeling population)

Pick the population again the same way as the earlier notebooks (top 20% by early-window spending).

In [2]:
early_window = tx_stable[
    (tx_stable["WEEK_NO"] >= STABLE_MIN_WEEK) &
    (tx_stable["WEEK_NO"] < STABLE_MIN_WEEK + EARLY_N)
].copy()

sales_early_full = early_window.groupby("HOUSEHOLD_KEY")["SALES_VALUE"].sum()

panel_wide = pd.DataFrame(index=stable_households)
panel_wide["sales_early"] = sales_early_full.reindex(panel_wide.index).fillna(0)

cutoff_early = panel_wide["sales_early"].quantile(1 - TOP_X_PCT/100)
panel_wide["group_early"] = np.where(panel_wide["sales_early"] >= cutoff_early, "top 20%", "remaining 80%")

top20_households = panel_wide[panel_wide["group_early"] == "top 20%"].index
print(f"Top 20% households on the early-window basis: {len(top20_households):,}")
print(f"Cutoff spending: {cutoff_early:.2f}")


Top 20% households on the early-window basis: 499
Cutoff spending: 535.29


## 2. Create the label: did the household stay in the top 20% on the late window too

Recompute the top 20% cutoff over the whole population using late-window (weeks 90-99) spending,
and check whether the households that were originally in the top 20% are still inside it in the late window.

In [3]:
late_window = tx_stable[
    (tx_stable["WEEK_NO"] > STABLE_MAX_WEEK - EARLY_N) &
    (tx_stable["WEEK_NO"] <= STABLE_MAX_WEEK)
].copy()

sales_late_full = late_window.groupby("HOUSEHOLD_KEY")["SALES_VALUE"].sum()
panel_wide["sales_late"] = sales_late_full.reindex(panel_wide.index).fillna(0)

cutoff_late = panel_wide["sales_late"].quantile(1 - TOP_X_PCT/100)
panel_wide["group_late"] = np.where(panel_wide["sales_late"] >= cutoff_late, "top 20%", "remaining 80%")

print(f"Top 20% cutoff spending on the late-window basis: {cutoff_late:.2f}")

# Label: among households originally in the top 20%, 0 (stayed) if still in the top 20% in the late window, otherwise 1 (declined/dropped out)
labels = panel_wide.loc[top20_households, "group_late"].apply(
    lambda x: 0 if x == "top 20%" else 1
)
labels.name = "label_shrink"

print("\nLabel distribution (0 = stayed, 1 = declined/dropped out)")
print(labels.value_counts())
print(f"Decline rate: {labels.mean()*100:.1f}%")


Top 20% cutoff spending on the late-window basis: 627.88

Label distribution (0 = stayed, 1 = declined/dropped out)
label_shrink
0    276
1    223
Name: count, dtype: int64
Decline rate: 44.7%


In [4]:
# --- source CSV: label definition (top 20% in early window -> still top 20% in late window?)
pd.DataFrame([{
    "n_stable_households": len(stable_households), "n_top20_early": len(top20_households),
    "cutoff_early": cutoff_early, "n_tied_at_cutoff_early": int((panel_wide["sales_early"] == cutoff_early).sum()),
    "cutoff_late": cutoff_late, "n_tied_at_cutoff_late": int((panel_wide["sales_late"] == cutoff_late).sum()),
    "n_retained": int((labels == 0).sum()), "n_shrink": int((labels == 1).sum()),
    "shrink_rate_pct": labels.mean() * 100, "retained_rate_pct": (1 - labels.mean()) * 100,
}]).to_csv(OUT_DIR / "ml_label_summary.csv", index=False)

## 3. Features 1-2: visit frequency, mean items per basket (transaction_data)

In [5]:
early_top20 = early_window[early_window["HOUSEHOLD_KEY"].isin(top20_households)].copy()

# avg_qty_per_basket based on the QUANTITY sum is not used, because weight/volume-unit products (meat, bulk, etc.)
# mix in severe extreme values (e.g. QUANTITY=41,686).
# Instead use "mean number of distinct products per basket (item diversity)", which is free of the unit problem.

feat_visit = early_top20.groupby("HOUSEHOLD_KEY").agg(
    n_baskets=("BASKET_ID", "nunique"),
).reset_index().set_index("HOUSEHOLD_KEY")

items_per_basket = (
    early_top20.groupby(["HOUSEHOLD_KEY", "BASKET_ID"])["PRODUCT_ID"]
    .nunique()
    .reset_index()
)
avg_items = items_per_basket.groupby("HOUSEHOLD_KEY")["PRODUCT_ID"].mean().rename("avg_items_per_basket")

feat_visit = feat_visit.join(avg_items)

print(feat_visit.describe())
feat_visit.head()


        n_baskets  avg_items_per_basket
count  499.000000            499.000000
mean    29.332665             13.507945
std     23.109256              8.263343
min      4.000000              1.208333
25%     16.000000              8.057143
50%     23.000000             11.238095
75%     35.000000             16.572917
max    184.000000             48.750000


,n_baskets,avg_items_per_basket
HOUSEHOLD_KEY,,
6,30,9.500000
13,36,9.083333
17,15,6.133333
18,15,13.333333
19,25,12.880000


## 4. Feature 3: category diversity (product join)

In [6]:
early_top20_prod = early_top20.merge(
    product[["PRODUCT_ID", "COMMODITY_DESC"]], on="PRODUCT_ID", how="left"
)

feat_diversity = early_top20_prod.groupby("HOUSEHOLD_KEY")["COMMODITY_DESC"].nunique()
feat_diversity.name = "category_diversity"

print(feat_diversity.describe())


count    499.000000
mean      83.715431
std       20.618235
min        5.000000
25%       70.000000
50%       82.000000
75%       96.000000
max      158.000000
Name: category_diversity, dtype: float64


## 5. Feature 4: share of purchases of display/mailer-exposed products (causal_data join)

Filtering causal_data to the early window (weeks 17-26) only, flag whether each transaction line was a product-store-week combination that was exposed (display or mailer other than '0'),
and compute the per-household average (the exposure share).

In [7]:
causal_early = causal[
    (causal["WEEK_NO"] >= STABLE_MIN_WEEK) &
    (causal["WEEK_NO"] < STABLE_MIN_WEEK + EARLY_N)
][["PRODUCT_ID", "STORE_ID", "WEEK_NO", "DISPLAY", "MAILER"]].copy()

# Unify the code values as strings (avoids 0 vs '0' comparison errors)
causal_early["DISPLAY"] = causal_early["DISPLAY"].astype(str)
causal_early["MAILER"] = causal_early["MAILER"].astype(str)

early_top20_causal = early_top20.merge(
    causal_early, on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"], how="left"
)

# Unmatched transactions (not in causal_data) have no exposure information, so they are treated as not exposed
early_top20_causal["DISPLAY"] = early_top20_causal["DISPLAY"].fillna("0")
early_top20_causal["MAILER"] = early_top20_causal["MAILER"].fillna("0")

early_top20_causal["exposed"] = (
    (early_top20_causal["DISPLAY"] != "0") | (early_top20_causal["MAILER"] != "0")
).astype(int)

feat_exposure = early_top20_causal.groupby("HOUSEHOLD_KEY")["exposed"].mean()
feat_exposure.name = "exposure_ratio"

print(feat_exposure.describe())


count    499.000000
mean       0.217594
std        0.078652
min        0.000000
25%        0.162174
50%        0.212121
75%        0.266591
max        0.462963
Name: exposure_ratio, dtype: float64


## 6. Features 5-10: demographics (hh_demographic, missing values as 'No record')

In [8]:
demo_cols = ["AGE_DESC", "INCOME_DESC", "HOMEOWNER_DESC", "HH_COMP_DESC",
             "HOUSEHOLD_SIZE_DESC", "KID_CATEGORY_DESC"]

feat_demo = hh_demo.set_index("HOUSEHOLD_KEY")[demo_cols].reindex(top20_households)
feat_demo = feat_demo.fillna("No record")

print("Check the distribution of each column after handling missing values (No record)")
for c in demo_cols:
    print(f"\n--- {c} ---")
    print(feat_demo[c].value_counts())


Check the distribution of each column after handling missing values (No record)

--- AGE_DESC ---
AGE_DESC
No record    180
45-54        118
35-44         91
25-34         58
55-64         20
65+           19
19-24         13
Name: count, dtype: int64

--- INCOME_DESC ---
INCOME_DESC
No record    180
50-74K        84
35-49K        54
75-99K        36
25-34K        25
Under 15K     24
15-24K        23
125-149K      21
100-124K      19
150-174K      18
250K+          8
175-199K       5
200-249K       2
Name: count, dtype: int64

--- HOMEOWNER_DESC ---
HOMEOWNER_DESC
Homeowner          216
No record          180
Unknown             79
Renter              19
Probable Owner       3
Probable Renter      2
Name: count, dtype: int64

--- HH_COMP_DESC ---
HH_COMP_DESC
No record           180
2 Adults No Kids    108
2 Adults Kids        82
Single Female        47
Single Male          35
Unknown              26
1 Adult Kids         21
Name: count, dtype: int64

--- HOUSEHOLD_SIZE_DESC ---
HOUSEHO

## 7. Combine the whole feature table

In [9]:
feature_table = panel_wide.loc[top20_households, ["sales_early"]].copy()
feature_table = feature_table.join(feat_visit)
feature_table = feature_table.join(feat_diversity)
feature_table = feature_table.join(feat_exposure)
feature_table = feature_table.join(feat_demo)
feature_table = feature_table.join(labels)
feature_table = feature_table.sort_index()  # fixed row order -> reproducible CV folds

print(f"Final feature table shape: {feature_table.shape}")
print("\nMissing-value check (should be 0; demographics were already filled with 'No record')")
print(feature_table.isna().sum())

feature_table.head(10)


Final feature table shape: (499, 12)

Missing-value check (should be 0; demographics were already filled with 'No record')
sales_early             0
n_baskets               0
avg_items_per_basket    0
category_diversity      0
exposure_ratio          0
AGE_DESC                0
INCOME_DESC             0
HOMEOWNER_DESC          0
HH_COMP_DESC            0
HOUSEHOLD_SIZE_DESC     0
KID_CATEGORY_DESC       0
label_shrink            0
dtype: int64


,sales_early,n_baskets,avg_items_per_basket,category_diversity,exposure_ratio,AGE_DESC,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,label_shrink
6,880.36,30,9.500000,80,0.203509,No record,No record,No record,No record,No record,No record,1
13,1899.43,36,9.083333,89,0.293578,25-34,75-99K,Homeowner,2 Adults Kids,4,2,0
17,554.76,15,6.133333,43,0.173913,65+,Under 15K,Homeowner,2 Adults No Kids,2,None/Unknown,1
18,678.38,15,13.333333,73,0.220000,45-54,100-124K,Homeowner,2 Adults No Kids,2,None/Unknown,0
19,762.90,25,12.880000,89,0.400621,35-44,15-24K,Unknown,Single Female,1,None/Unknown,0
36,662.85,50,3.780000,52,0.169312,No record,No record,No record,No record,No record,No record,0
39,945.71,10,32.800000,91,0.167683,35-44,125-149K,Homeowner,2 Adults No Kids,2,None/Unknown,1
40,1390.98,35,13.914286,123,0.137577,45-54,Under 15K,Homeowner,2 Adults No Kids,2,None/Unknown,0
52,664.38,20,11.600000,91,0.176724,45-54,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,0
58,1145.20,34,7.794118,95,0.245283,25-34,150-174K,Homeowner,2 Adults Kids,4,2,0


## 8. Basic statistics of the features by label (exploration before encoding)

Before the actual encoding and modeling, check in advance whether the numeric features differ noticeably by label (stayed vs declined).

In [10]:
numeric_cols = ["sales_early", "n_baskets", "avg_items_per_basket", "category_diversity", "exposure_ratio"]

print(feature_table.groupby("label_shrink")[numeric_cols].agg(["mean", "median"]))


              sales_early          n_baskets        avg_items_per_basket            category_diversity        exposure_ratio          
                     mean  median       mean median                 mean     median               mean median           mean    median
label_shrink                                                                                                                          
0             1046.155399  918.32  29.369565   24.0            14.173353  11.746429          88.246377   87.0       0.213161  0.205941
1              825.776502  713.05  29.286996   22.0            12.684391  10.230769          78.107623   77.0       0.223079  0.222222


## 9. Save (reused in the next steps: encoding -> VIF -> modeling)

In [11]:
feature_table.index.name = "HOUSEHOLD_KEY"
feature_table.to_csv(INTERMEDIATE_DIR / "h4_shrink_feature_table_raw.csv")
print("Saved: outputs/intermediate/h4_shrink_feature_table_raw.csv")
print(f"shape: {feature_table.shape}")


Saved: outputs/intermediate/h4_shrink_feature_table_raw.csv
shape: (499, 12)


In [12]:
# --- source CSVs: demographic coverage, numeric features by label, QUANTITY check
_no_demo = int((feat_demo == "No record").all(axis=1).sum())
pd.DataFrame([{
    "n_top20": len(feature_table), "n_without_demo": _no_demo, "n_with_demo": len(feature_table) - _no_demo,
    "without_demo_pct": _no_demo / len(feature_table) * 100,
}]).to_csv(OUT_DIR / "ml_demo_missing.csv", index=False)
_byl = feature_table.groupby("label_shrink")[numeric_cols].agg(["mean", "median"])
_byl.columns = ["_".join(c) for c in _byl.columns]
_byl.reset_index().to_csv(OUT_DIR / "ml_feature_by_label.csv", index=False)
_rows = []
_hh_qty = early_top20.groupby("HOUSEHOLD_KEY")["QUANTITY"].sum().nlargest(10)
_rows += [{"metric": "household_quantity_sum_early_window", "rank": r + 1, "household_key": k, "value": v}
          for r, (k, v) in enumerate(_hh_qty.items())]
_line = early_top20.nlargest(10, "QUANTITY")[["HOUSEHOLD_KEY", "QUANTITY"]]
_rows += [{"metric": "single_line_quantity_early_window", "rank": r + 1, "household_key": k, "value": v}
          for r, (k, v) in enumerate(zip(_line["HOUSEHOLD_KEY"], _line["QUANTITY"]))]
_qpb = (early_top20.groupby("HOUSEHOLD_KEY")["QUANTITY"].sum() / feat_visit["n_baskets"]).nlargest(10)
_rows += [{"metric": "avg_qty_per_basket_early_window", "rank": r + 1, "household_key": k, "value": v}
          for r, (k, v) in enumerate(_qpb.items())]
_mx = tx.loc[tx["QUANTITY"].idxmax()]
_rows.append({"metric": "max_line_quantity_all_transactions", "rank": 1, "household_key": _mx["HOUSEHOLD_KEY"], "value": _mx["QUANTITY"]})
_q = pd.DataFrame(_rows)
_q.to_csv(INTERMEDIATE_DIR / "ml_quantity_check.csv", index=False)  # has household_key -> intermediate
_q.drop(columns="household_key").to_csv(OUT_DIR / "ml_quantity_summary.csv", index=False)  # aggregate, tracked